# 03 - Unsupervised Clustering & Anomaly Detection
## AI-Based Cyber Risk Detection System

This notebook investigates unsupervised methods to detect novel or zero-day cyber threats:
1. **K-Means Clustering** with Silhouette & Elbow analysis
2. **Hierarchical (Agglomerative) Clustering**
3. **DBSCAN Density-Based Clustering**
4. **Isolation Forest** outlier detection
5. **Local Outlier Factor (LOF)** density-based anomaly scoring
6. Cluster validation: Silhouette score, Calinski-Harabasz, and Davies-Bouldin index.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from src.pipeline import CyberRiskPipeline
from src.anomaly.clustering import ClusteringAnalyzer
from src.anomaly.detectors import AnomalyDetector

pipeline = CyberRiskPipeline()
pipeline.run_data_pipeline(use_synthetic=True)
X = pipeline.X_train[:1500]  # Subset for interactive exploration

print(f"Feature matrix loaded: {X.shape}")

### 1. K-Means Clustering & Metrics

In [ ]:
clustering = ClusteringAnalyzer()
kmeans_res = clustering.kmeans(X, n_clusters=4)
print("K-Means Quality Metrics:")
for k, v in kmeans_res['metrics'].items():
    print(f"  {k}: {v:.4f}")

### 2. Hierarchical & DBSCAN Clustering

In [ ]:
hier_res = clustering.hierarchical(X, n_clusters=4)
dbscan_res = clustering.dbscan(X, eps=0.6, min_samples=5)

print(f"DBSCAN Clusters: {dbscan_res['n_clusters']} | Noise Outliers: {dbscan_res['n_noise']}")

### 3. Isolation Forest & Local Outlier Factor (LOF)

In [ ]:
detector = AnomalyDetector()
if_res = detector.isolation_forest(X, contamination=0.08)
lof_res = detector.local_outlier_factor(X, contamination=0.08)

print(f"Isolation Forest Anomalies: {if_res['n_anomalies']} ({if_res['n_anomalies']/len(X)*100:.1f}%)")
print(f"LOF Anomalies: {lof_res['n_anomalies']} ({lof_res['n_anomalies']/len(X)*100:.1f}%)")

### 4. Visualizing Anomalies in 2D PCA Space

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(10, 6))
plt.scatter(X_pca[if_res['predictions'] == 1, 0], X_pca[if_res['predictions'] == 1, 1], c='lightgray', label='Normal Samples', alpha=0.6)
plt.scatter(X_pca[if_res['predictions'] == -1, 0], X_pca[if_res['predictions'] == -1, 1], c='crimson', label='Detected Anomalies (Isolation Forest)', s=30)
plt.title('Unsupervised Anomaly Identification (PCA Projection)')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()